## Professionally You!

1. Add GitHub Live

In [ ]:
!uv pip install apify-client

Using Python 3.12.13 environment at: C:\Users\sirpr\OneDrive\Documents\GitHub\agents\.venv
Resolved 5 packages in 422ms
 Downloaded impit
Prepared 3 packages in 439ms
Installed 3 packages in 123ms
 + apify-client==2.5.0
 + apify-shared==2.2.0
 + impit==0.12.0


In [5]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr
from github import Github

# RAG specific imports
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [7]:
load_dotenv(override=True)
openai = OpenAI()

# Professional Identity
name = "Sirpreet Dhillon"

In [8]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


In [9]:
documents_to_load = [
    {"path": "me/linkedin.pdf", "type": "pdf"},
    {"path": "me/resume.pdf", "type": "pdf"},
    {"path": "me/projects.pdf", "type": "pdf"},
    {"path": "me/summary.txt", "type": "txt"}
]

In [10]:
all_docs = []
for doc in documents_to_load:
    if os.path.exists(doc["path"]):
        loader = PyPDFLoader(doc["path"]) if doc["type"] == "pdf" else TextLoader(doc["path"])
        all_docs.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(all_docs)

vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def get_rag_context(query):
    relevant_docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in relevant_docs])

In [11]:
def fetch_github_projects():
    """Fetches real-time project data from GitHub."""
    try:
        g = Github(os.getenv("GITHUB_TOKEN"))
        repos = [{"name": r.name, "url": r.html_url, "stars": r.stargazers_count} 
                 for r in g.get_user().get_repos(sort="updated")[:3]]
        return json.dumps(repos)
    except Exception as e:
        return f"Error fetching GitHub: {str(e)}"

In [12]:
def send_lead_to_pushover(user_name, user_email, message):
    """Sends user contact info to Sirpreet's phone via Pushover."""
    url = "https://api.pushover.net/1/messages.json"
    formatted_msg = f"New Lead!\nName: {user_name}\nEmail: {user_email}\nMessage: {message}"
    
    data = {
        "token": os.getenv("PUSHOVER_TOKEN"),
        "user": os.getenv("PUSHOVER_USER"),
        "message": formatted_msg
    }
    response = requests.post(url, data=data)
    return "Message sent to Sirpreet successfully!" if response.status_code == 200 else "Failed to send."

In [13]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "fetch_github_projects",
            "description": "Get real-time info about my latest GitHub repositories."
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_lead_to_pushover",
            "description": "Call this when a user wants to contact Sirpreet or leave their details.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_name": {"type": "string", "description": "The user's name"},
                    "user_email": {"type": "string", "description": "The user's email address"},
                    "message": {"type": "string", "description": "The message they want to leave"}
                },
                "required": ["user_name", "user_email", "message"]
            }
        }
    }
]

In [14]:
def evaluate_response(user_query, ai_response):
    eval_prompt = f"You are a professional brand manager. Ensure the following response is polite and professional for {name}.\nDraft: {ai_response}\nRespond 'PASS' or 'CORRECTION: [instructions]'."
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": eval_prompt}]
    )
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    context = get_rag_context(message)
    # system_message = f"You are {name}. Use this context: {context}. If a user wants to reach out, ask for their name and email, then use 'send_lead_to_pushover'."
    
    system_message = f"""
    You are acting as {name}. You are a professional, knowledgeable, and helpful 
    version of myself designed to interact with recruiters, collaborators, and peers.
    
    YOUR CORE INSTRUCTIONS:
    - Maintain a professional yet approachable tone.
    - Use the following retrieved context to provide specific details about my 
      experience, education, and skills:
      {context}
    
    - If a user asks something not covered in the context, do not make up facts.
    - Instead, politely offer to record their question or contact details 
      using the 'send_lead_to_pushover' tool so I can get back to them personally.
    - If they want to see my live work, use the 'fetch_github_projects' tool.
    
    (Add any other specific personality rules you had in your original prompt here)
    """
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    # Tool Execution Loop
    while True:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        choice = response.choices[0].message
        
        if not choice.tool_calls:
            break
            
        messages.append(choice)
        for tool_call in choice.tool_calls:
            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            
            if function_name == "fetch_github_projects":
                result = fetch_github_projects()
            elif function_name == "send_lead_to_pushover":
                result = send_lead_to_pushover(**args)
            
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    draft = response.choices[0].message.content
    eval_result = evaluate_response(message, draft)
    
    if "PASS" in eval_result:
        return draft
    else:
        # Self-Correction
        messages.append({"role": "user", "content": f"Correction: {eval_result}"})
        final = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
        return final.choices[0].message.content

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unknown_question
Push: Recording What is your favourite singer? asked that I couldn't answer


## And now for deployment

This code is in `app.py`

We will deploy to HuggingFace Spaces.

Before you start: remember to update the files in the "me" directory - your LinkedIn profile and summary.txt - so that it talks about you! Also change `self.name = "Ed Donner"` in `app.py`..  

Also check that there's no README file within the 1_foundations directory. If there is one, please delete it. The deploy process creates a new README file in this directory for you.

1. Visit https://huggingface.co and set up an account  
2. From the Avatar menu on the top right, choose Access Tokens. Choose "Create New Token". Give it WRITE permissions - it needs to have WRITE permissions! Keep a record of your new key.  
3. In the Terminal, run: `uv tool install 'huggingface_hub[cli]'` to install the HuggingFace tool, then `hf auth login --token YOUR_TOKEN_HERE`, like `hf auth login --token hf_xxxxxx`, to login at the command line with your key. Afterwards, run `hf auth whoami` to check you're logged in  
4. Take your new token and add it to your .env file: `HF_TOKEN=hf_xxx` for the future
5. From the 1_foundations folder, enter: `uv run gradio deploy` 
6. Follow its instructions: name it "career_conversation", specify app.py, choose cpu-basic as the hardware, say Yes to needing to supply secrets, provide your openai api key, your pushover user and token, and say "no" to github actions.  

Thank you Robert, James, Martins, Andras and Priya for these tips.  
Please read the next 2 sections - how to change your Secrets, and how to redeploy your Space (you may need to delete the README.md that gets created in this 1_foundations directory).

#### More about these secrets:

If you're confused by what's going on with these secrets: it just wants you to enter the key name and value for each of your secrets -- so you would enter:  
`OPENAI_API_KEY`  
Followed by:  
`sk-proj-...`  

And if you don't want to set secrets this way, or something goes wrong with it, it's no problem - you can change your secrets later:  
1. Log in to HuggingFace website  
2. Go to your profile screen via the Avatar menu on the top right  
3. Select the Space you deployed  
4. Click on the Settings wheel on the top right  
5. You can scroll down to change your secrets (Variables and Secrets section), delete the space, etc.

#### And now you should be deployed!

If you want to completely replace everything and start again with your keys, you may need to delete the README.md that got created in this 1_foundations folder.

Here is mine: https://huggingface.co/spaces/ed-donner/Career_Conversation

I just got a push notification that a student asked me how they can become President of their country 😂😂

For more information on deployment:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

To delete your Space in the future:  
1. Log in to HuggingFace
2. From the Avatar menu, select your profile
3. Click on the Space itself and select the settings wheel on the top right
4. Scroll to the Delete section at the bottom
5. ALSO: delete the README file that Gradio may have created inside this 1_foundations folder (otherwise it won't ask you the questions the next time you do a gradio deploy)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• First and foremost, deploy this for yourself! It's a real, valuable tool - the future resume..<br/>
            • Next, improve the resources - add better context about yourself. If you know RAG, then add a knowledge base about you.<br/>
            • Add in more tools! You could have a SQL database with common Q&A that the LLM could read and write from?<br/>
            • Bring in the Evaluator from the last lab, and add other Agentic patterns.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your career alter-ego) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>